# L6 — Funzioni e cicli: automatizzare (SOLUZIONI)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrealorenzon/SIMA2026/blob/soluzioni/L06_funzioni_cicli/L06_soluzioni.ipynb)

## Promemoria

- **Shift + Invio** esegue una cella. **File → Salva una copia in Drive** per non perdere il lavoro.
- Per ogni passo trovi 🔎 **cosa cercare**. Gemini (✨) è disponibile, ma prova prima a capire cosa ti serve.
- ⏱️ Bloccato da più di 10 minuti? Chiedi.
- 👥 Si lavora in **gruppi da 3**, con tre ruoli che **ruotano a ogni passo**: chi **scrive** (condivide lo schermo e scrive il codice), chi **cerca** (Google, Gemini, dispense), chi **controlla** (il risultato ha senso? il numero torna?).
- Se fai un pasticcio: **Runtime → Riavvia ed esegui tutto**.

## Contesto

Ti arrivano **20 file Excel**, uno per regione, tutti con la stessa struttura. Per ognuno vuoi lo stesso riepilogo: numero di imprese, fatturato mediano, produttività mediana. In Excel vorrebbe dire aprire 20 file e rifare 20 volte gli stessi passaggi. In Python scrivi il procedimento **una volta** (una funzione) e lo ripeti su tutti i file (un ciclo).

## 0. Preparazione

Esegui questa cella così com'è: scarica i file del corso in Colab. Puoi vederli nel pannello dei file (icona della cartella a sinistra), in `SIMA2026/dati/per_regione`.

In [ ]:
# scarica la cartella del corso (solo se manca)
![ -d SIMA2026 ] || git clone -q --depth 1 https://github.com/andrealorenzon/SIMA2026.git

from pathlib import Path
import pandas as pd

cartella = Path("SIMA2026/dati/per_regione")
print(cartella.exists())

## Parte 1 — Una funzione

### 1. Prima a mano

Carica il file della Lombardia (`cartella / "Lombardia.xlsx"`) e calcola: numero di imprese diverse, fatturato mediano, produttività mediana (fatturato diviso addetti).

🔎 *Da cercare:* `pandas read_excel` · `pandas nunique` · `pandas median`

In [ ]:
lomb = pd.read_excel(cartella / "Lombardia.xlsx")
print(lomb["id_impresa"].nunique())
print(lomb["fatturato"].median())
print((lomb["fatturato"] / lomb["addetti"]).median())

### 2. La tua prima funzione

Scrivi una funzione `riepilogo(percorso)` che fa le stesse cose per un file qualsiasi e **restituisce** i risultati in un dizionario, con anche il nome della regione (il nome del file senza `.xlsx`). Provala sulla Lombardia: i numeri devono essere gli stessi del passo 1.

🔎 *Da cercare:* `python def return` · `python dizionario` · `pathlib stem`

In [ ]:
def riepilogo(percorso):
    df = pd.read_excel(percorso)
    return {
        "regione": Path(percorso).stem,
        "imprese": df["id_impresa"].nunique(),
        "fatturato_mediano": df["fatturato"].median(),
        "produttivita_mediana": (df["fatturato"] / df["addetti"]).median(),
    }

riepilogo(cartella / "Lombardia.xlsx")

### 3. Riusala

Usa la funzione sul Veneto e sulla Sicilia. Quante righe di codice hai scritto in più?

In [ ]:
print(riepilogo(cartella / "Veneto.xlsx"))
print(riepilogo(cartella / "Sicilia.xlsx"))

## Parte 2 — Un ciclo

### 4. Tutti i file

Crea la lista di tutti i file `.xlsx` della cartella, in ordine alfabetico. Quanti sono?

🔎 *Da cercare:* `pathlib glob` · `python sorted` · `python len`

In [ ]:
file = sorted(cartella.glob("*.xlsx"))
len(file)

### 5. Il primo ciclo

Con un ciclo `for`, stampa il nome di ogni regione (il nome del file, senza `.xlsx`).

🔎 *Da cercare:* `python for` · `pathlib stem`

In [ ]:
for f in file:
    print(f.stem)

### 6. Il riepilogo di tutte

Con un ciclo, applica `riepilogo` a ogni file e raccogli i risultati in una lista. Funziona? Se c'è un errore: **su quale file** si ferma, e perché?

🔎 *Da cercare:* `python list append` · l'ultima riga dell'errore

In [ ]:
risultati = []
for f in file:
    print(f.stem)
    try:
        risultati.append(riepilogo(f))
    except KeyError as e:
        print("   ERRORE in", f.name, "->", e)

*Risposta:* Si ferma sul Molise con `KeyError: 'fatturato'`: in quel file la colonna si chiama `Fatturato`, con la maiuscola. Trucco per trovarlo: stampare il nome del file **prima** di elaborarlo; l'ultimo nome stampato è quello che ha dato errore. (Nella soluzione l'errore è intercettato con `try`/`except` solo per mostrarlo senza fermare il notebook.)

### 7. Rendi la funzione robusta

Modifica `riepilogo` in modo che, appena letto il file, trasformi in minuscolo tutti i nomi delle colonne. Poi rifai il ciclo: deve elaborare tutti i 20 file. Trasforma la lista dei risultati in un DataFrame.

🔎 *Da cercare:* `pandas columns str lower` · `pandas DataFrame da lista di dizionari`

In [ ]:
def riepilogo(percorso):
    df = pd.read_excel(percorso)
    df.columns = df.columns.str.lower()
    return {
        "regione": Path(percorso).stem,
        "imprese": df["id_impresa"].nunique(),
        "fatturato_mediano": df["fatturato"].median(),
        "produttivita_mediana": (df["fatturato"] / df["addetti"]).median(),
    }

risultati = []
for f in file:
    risultati.append(riepilogo(f))

tab = pd.DataFrame(risultati)
print(len(tab))
tab.head()

### 8. Il report

Ordina la tabella per fatturato mediano, dal più alto, e salvala in un file Excel `riepilogo_regioni.xlsx`.

🔎 *Da cercare:* `pandas sort_values` · `pandas to_excel index`

In [ ]:
tab = tab.sort_values("fatturato_mediano", ascending=False)
tab.to_excel("riepilogo_regioni.xlsx", index=False)
tab.head()

## Domanda finale

Quanto tempo ti avrebbe richiesto lo stesso lavoro in Excel, su 20 file? E se domani i file diventano 200, cosa cambia nel tuo codice?

*Risposta:* In Excel: aprire 20 file e ripetere gli stessi calcoli, con il rischio di sbagliarne uno. Nel codice: niente, il ciclo prende tutti i file che trova nella cartella. Ed è proprio un file "diverso dagli altri" (il Molise) che in Excel sarebbe passato inosservato.

---
## ⭐ Bonus (per chi ha finito)

### B1. Un parametro in più

Aggiungi a `riepilogo` un parametro `anno`: se viene indicato, il riepilogo considera solo quell'anno. Se non viene indicato, li considera tutti. Prova `riepilogo(cartella / "Lazio.xlsx", anno=2020)`.

🔎 *Da cercare:* `python parametro valore predefinito None`

In [ ]:
def riepilogo(percorso, anno=None):
    df = pd.read_excel(percorso)
    df.columns = df.columns.str.lower()
    if anno is not None:
        df = df[df["anno"] == anno]
    return {
        "regione": Path(percorso).stem,
        "imprese": df["id_impresa"].nunique(),
        "fatturato_mediano": df["fatturato"].median(),
        "produttivita_mediana": (df["fatturato"] / df["addetti"]).median(),
    }

print(riepilogo(cartella / "Lazio.xlsx"))
print(riepilogo(cartella / "Lazio.xlsx", anno=2020))

### B2. Ricomponi il dataset

Con un ciclo, leggi tutti i 20 file, aggiungi a ognuno una colonna `regione` con il nome del file, e mettili tutti in fila in un unico DataFrame. Quante righe ottieni? Quante dovrebbero essere?

🔎 *Da cercare:* `pandas concat` · `pandas columns str lower`

In [ ]:
pezzi = []
for f in file:
    d = pd.read_excel(f)
    d.columns = d.columns.str.lower()
    d["regione"] = f.stem
    pezzi.append(d)
tutto = pd.concat(pezzi)
print(tutto.shape)

*Risposta:* 2.500 righe, come `imprese_pulito.xlsx`. Senza la correzione delle maiuscole, il Molise avrebbe creato una colonna `Fatturato` separata, piena di valori mancanti per tutte le altre regioni.

---
## Compito a casa

- [Kaggle Learn – Python](https://www.kaggle.com/learn/python), lezioni **Functions and Getting Help** e **Loops and List Comprehensions** (~40 min)
- Facoltativo: [QuantEcon – Functions](https://python-programming.quantecon.org/functions.html)